In [1]:
import torch

print("Đã cài đặt thành công PyTorch phiên bản:", torch.__version__)
print("Apple Silicon (MPS) đã sẵn sàng chưa:", torch.backends.mps.is_available())

Đã cài đặt thành công PyTorch phiên bản: 2.8.0
Apple Silicon (MPS) đã sẵn sàng chưa: True


In [8]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Định nghĩa Data Augmentation
IMG_SIZE = 224
BATCH_SIZE = 32

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224), # Cắt ngẫu nhiên một góc và phóng to
        transforms.RandomHorizontalFlip(p=0.5), # 50% cơ hội lật ngược ảnh (như soi gương)
        transforms.RandomRotation(degrees=15), # Xoay nghiêng ảnh từ -15 đến 15 độ
        transforms.ColorJitter(brightness=0.2, contrast=0.2), # Thay đổi độ sáng/tương phản
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([ 
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# 2. Trỏ đúng tên thư mục là 'dataset'
data_dir = 'dataset' 

# 3. Tải dữ liệu lên
image_datasets = {
    x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
    for x in ['train', 'val'] 
}

# 4. Đóng gói thành các Batch
dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=True if x == 'train' else False)
    for x in ['train', 'val'] 
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f"Các nhãn phân loại (Classes): {class_names}")
print(f"Số lượng ảnh Train: {dataset_sizes['train']}")
print(f"Số lượng ảnh Validation: {dataset_sizes['val']}")

Các nhãn phân loại (Classes): ['fake', 'real']
Số lượng ảnh Train: 90409
Số lượng ảnh Validation: 21776


In [9]:
import torch 
import time
import copy
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler # <-- MỚI THÊM: Import thư viện giảm tốc độ học
from torchvision import models

# Kiểm tra lại thiết bị (đảm bảo dùng Apple Silicon MPS)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# 1. Tải mô hình ResNet-50 đã học trước trên ImageNet
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# 2. Đóng băng tất cả các lớp (Feature Extraction)
for param in model.parameters():
    param.requires_grad = False

# 3. Chỉnh sửa lớp cuối cùng (Classification Head)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.6), # <-- SỬA: Tăng Dropout lên 0.6 (đánh ngất 60% nơ-ron ép nó học nghiêm túc)
    nn.Linear(512, 2) 
)

# Đưa mô hình vào GPU/MPS
model = model.to(device)

# 4. Cài đặt Hàm mất mát (Loss) và Thuật toán tối ưu (Optimizer)
criterion = nn.CrossEntropyLoss()

# <-- SỬA: Thêm weight_decay=1e-4 vào thuật toán Adam (đeo tạ cho các nơ-ron)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001, weight_decay=1e-4)

# <-- MỚI THÊM: Cài đặt phanh thông minh (cứ qua 2 vòng học thì đi chậm lại 1 nửa)
step_lr_scheduler = lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

print("Đã tải và cấu hình xong mô hình ResNet-50 phiên bản chống Học Vẹt!")

Đã tải và cấu hình xong mô hình ResNet-50 phiên bản chống Học Vẹt!


In [10]:
import time
import copy
import torch

# <-- SỬA: Thêm chữ 'scheduler' vào trong ngoặc để hàm nhận cái phanh
def train_model(model, criterion, optimizer, scheduler, num_epochs=5):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  
            else:
                model.eval()   

            running_loss = 0.0
            running_corrects = 0

            # Lặp qua từng lô (batch) dữ liệu
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            # --- MỚI THÊM: BÓP PHANH Ở ĐÂY ---
            # Sau khi nó cày xong toàn bộ ảnh của pha 'train', ta giảm tốc độ học xuống
            if phase == 'train':
                scheduler.step()
            # ---------------------------------

            epoch_loss = running_loss / dataset_sizes[phase]
            # Đã sửa lỗi Apple Silicon: Dùng .float() thay vì .double()
            epoch_acc = running_corrects.float() / dataset_sizes[phase]

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Huấn luyện xong trong {time_elapsed // 60:.0f} phút {time_elapsed % 60:.0f} giây')
    print(f'Độ chính xác Validation tốt nhất: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model

In [11]:
import torch.optim as optim
from torch.optim import lr_scheduler

# 1. Mở khóa lớp sâu nhất (layer4) để AI học thêm vân tay của Midjourney/Flux
for param in model.layer4.parameters():
    param.requires_grad = True

# 2. Tạo động cơ mới (Optimizer) cho pha Fine-Tuning
optimizer_fine_tune = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-5}, # Lớp sâu thì học từ từ (1e-5)
    {'params': model.fc.parameters(), 'lr': 1e-4, 'weight_decay': 1e-4} # Giữ lại tạ chống học vẹt
])

# 3. CHẾ TẠO PHANH MỚI CHO ĐỘNG CƠ MỚI
scheduler_fine_tune = lr_scheduler.StepLR(optimizer_fine_tune, step_size=2, gamma=0.5)

print("🚀 Bắt đầu khóa huấn luyện Fine-Tuning đặc nhiệm...")

# 4. Huấn luyện (ĐÃ BỔ SUNG THÊM scheduler_fine_tune VÀO ĐÚNG VỊ TRÍ)
model_ft = train_model(model, criterion, optimizer_fine_tune, scheduler_fine_tune, num_epochs=5)

# 5. Tự động lưu mô hình ngay khi học xong
model_save_path = 'ai_image_detector.pth'
torch.save(model_ft.state_dict(), model_save_path)
print(f"🎉 Đã luyện đan xong! Lưu thành công tại: {model_save_path}")

🚀 Bắt đầu khóa huấn luyện Fine-Tuning đặc nhiệm...
Epoch 1/5
----------
Train Loss: 0.3624 Acc: 0.8315
Val Loss: 0.6028 Acc: 0.7141

Epoch 2/5
----------
Train Loss: 0.2457 Acc: 0.8957
Val Loss: 0.7150 Acc: 0.6970

Epoch 3/5
----------
Train Loss: 0.2058 Acc: 0.9132
Val Loss: 0.6738 Acc: 0.7253

Epoch 4/5
----------
Train Loss: 0.1908 Acc: 0.9236
Val Loss: 0.5641 Acc: 0.7797

Epoch 5/5
----------
Train Loss: 0.1755 Acc: 0.9277
Val Loss: 0.8228 Acc: 0.7181

Huấn luyện xong trong 122 phút 27 giây
Độ chính xác Validation tốt nhất: 0.779712
🎉 Đã luyện đan xong! Lưu thành công tại: ai_image_detector.pth
